In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
import numpy as np
import tensorflow as tf

train_df = pd.read_excel('https://github.com/pykwon/python/blob/master/testdata_utf8/hd_carprice.xlsx?raw=true', sheet_name = 'train')
test_df = pd.read_excel('https://github.com/pykwon/python/blob/master/testdata_utf8/hd_carprice.xlsx?raw=true', sheet_name='test')
# train_df = pd.read_excel('https://github.com/pykwon/python/blob/master/testdata_utf8/hd_carprice.xlsx')
# 주석의 코드 뒤에 ?raw=true랑 sheet 이름까지 넣엇어

print(train_df.head())
print(test_df.head())
print('-' * 100)

x_train = train_df.drop(['가격'], axis = 1)     # train의 feature는 가격을 뺀 나머지 데이터로 하자
x_test = test_df.drop(['가격'], axis = 1)       # test의 feature
y_train = train_df[['가격']]                    # train의 label
y_test = test_df[['가격']]                      # test의 label

print(x_train.head(2))
print(x_test.head(2))
print('-' * 100)
print(y_train.head(2))
print(y_test.head(2))
print('-' * 100)

print(x_train.columns)
print(x_train.shape)
print('-' * 100)

print(set(x_train.종류))            # {'대형', '중형', '준중형', '소형'}
print(set(x_train.연료))            # {'LPG', '가솔린', '디젤'}
print(set(x_train.변속기))          # {'자동', '수동'}
print('-' * 100)

# 종류, 연료, 변속기 열에 대해서는 LabelEncoder(), OneHotEncoder()를 적용
transformer = make_column_transformer((OneHotEncoder(), ['종류','연료','변속기']), remainder = 'passthrough')
# 위의 아규먼트에서     remainder = . . .의 디폴트값은
# remainder = 'drop'이래 (remainder 옵션이 나머지 열을 떨어뜨릴지(drop), 그대로 둘지(passthrough) 결정함.)
# 열이 transformer에 전달된대 근데 이건 지피티 돌려보자 못알아듣겟다

transformer.fit(x_train)
x_train = transformer.transform(x_train)        # 3개의 컬럼을 포함해 모든 컬럼이 표준화됨
x_test = transformer.transform(x_test)
print(x_train[:3], x_train.shape)               # (71, 16)
print(y_train[:3], y_train.shape)               # (71, 1)

     가격    년식   종류    연비   마력    토크   연료  하이브리드   배기량    중량 변속기
0  1885  2015  준중형  11.8  172  21.0  가솔린      0  1999  1300  자동
1  2190  2015  준중형  12.3  204  27.0  가솔린      0  1591  1300  자동
2  1135  2015   소형  15.0  100  13.6  가솔린      0  1368  1035  수동
3  1645  2014   소형  14.0  140  17.0  가솔린      0  1591  1090  자동
4  1960  2015   대형   9.6  175  46.0   디젤      0  2497  1990  자동
     가격    년식  종류    연비   마력    토크   연료  하이브리드   배기량    중량 변속기
0  1915  2015  대형   6.8  159  23.0  LPG      0  2359  1935  수동
1  1164  2012  소형  13.3  108  13.9  가솔린      0  1396  1035  자동
2  2817  2015  중형  14.4  184  41.0   디젤      0  1995  1792  자동
3  2160  2015  대형  10.9  175  46.0   디젤      0  2497  2210  수동
4  1915  2015  대형   6.4  159  23.0  LPG      0  2359  1935  자동
----------------------------------------------------------------------------------------------------
     년식   종류    연비   마력    토크   연료  하이브리드   배기량    중량 변속기
0  2015  준중형  11.8  172  21.0  가솔린      0  1999  1300  자동
1  2015  준중형  12.3  2

In [2]:
# 문제2)
# https://github.com/pykwon/python/tree/master/data
# 자전거 공유 시스템 분석용 데이터 train.csv를 이용하여 대여횟수에 영향을 주는 변수들을 골라 다중선형회귀분석 모델을 작성하시오.
# 모델 학습시에 발생하는 loss를 시각화하고 설명력을 출력하시오.
# 새로운 데이터를 input 함수를 사용해 키보드로 입력하여 대여횟수 예측결과를 콘솔로 출력하시오.

In [3]:
x_test=transformer.transform(x_test)
print(x_train[:2],' ',x_train.shape)
print(y_train.shape)

ValueError: X has 16 features, but ColumnTransformer is expecting 10 features as input.

In [1]:
# functional api 모델 작성
input=tf.keras.layers.Input(shape=(16,))
net tf.keras.layers.Dense(units=32,activation='relu')(input)
net tf.keras.layers.Dense(units=32,activation='relu')(net)
net tf.keras.layers.Dense(units=1,activation='relu')(net)
Input(shape=(16,))

# 입력 텐서 정의. 샘플당 특징 16개를 받는다는 뜻(배치 차원은 자동).
# Dense(32, activation='relu')(...) (두 번)
# 완전연결 은닉층 32유닛 + ReLU 비선형.
# 표현력↑, 비선형 경계 학습.
# 파라미터 수:
# 1층: (16+1)×32=544
# 2층: (32+1)×32=1056
# Dense(1, activation='relu')(net)
# 출력 1유닛. ReLU이므로 출력이 0 이상으로 제한됩니다.
# 파라미터: (32+1)×1=33

model=tf.keras.models.Model(input,net)
model.compile(optimizer='adam',loss='mean_squared_error',metrics=['mse'])
print(model.summary())

model.fit(x_train,y_train,epochs=50,validation_data=(x_test,y_test),verbose=2)
print('evaluate : ',model.evaluate(x_test,y_test))

SyntaxError: invalid syntax (429156867.py, line 3)

In [10]:
y_predict=model.predict(x_test)
print('예측값 : ',y_predict[:5].ravel())
print('실제값 : ',y_test[:5].values.ravel())

# 새로운 값으로 예측
new_x_test=[[2016,'대형',11.8,172,21.0,'가솔린',0,1999,1300,'자동']]
new_x_test=pd.DataFrame(new_x_test,columns=['년식','종류','연비','토크','연료','하이브리드','배기량','중량','변속기'])
print(new_x_test)

new_x_test=transformer.transform(new_x_test)
new_predict=model.predict(new_x_test)
print('예측값 : ',new_predict.ravel())

# regression은 파라미터가 복잡해질 이유가 없고, 약간의 relu를 걸고 linear만 적으며 잘돌아감

NameError: name 'model' is not defined